# Warm-up: Conditionals Light Board

Run the setup cell first. Then move the slider and press the buttons: the light shows **which branch of the code ran**.

`light()` works just like `print()`: give it a color, then the message.

The `%%slider` and `%%buttons` line at the top of a cell just adds the controls; the code underneath is a normal `if` statement.

In [ ]:
# @title ▶ Run this first (sets up the lights)
from IPython.display import display, HTML, clear_output
from IPython.core.magic import register_cell_magic
import ipywidgets as widgets
import time


def light(color, *message):
    """Show a colored indicator light with a message next to it."""
    text = ' '.join(str(m) for m in message)
    html = ('<div style="display:flex;align-items:center;gap:18px;'
            'font-family:sans-serif;font-size:22px;padding:8px 0">'
            '<div style="width:64px;height:64px;border-radius:50%;'
            'background:' + color + ';box-shadow:0 0 28px ' + color + ';'
            'border:3px solid #333"></div>'
            '<b>' + text + '</b></div>')
    clear_output(wait=True)
    display(HTML(html))


def wait(seconds=0.6):
    time.sleep(seconds)


live_controls = {}


def run_live(cell, controls):
    """Show the controls, and re-run the cell's code every time one changes."""
    live_controls.clear()
    live_controls.update(controls)
    out = widgets.Output()
    namespace = get_ipython().user_ns

    def update(change=None):
        for name in controls:
            namespace[name] = controls[name].value
        with out:
            clear_output(wait=True)
            exec(cell, namespace)

    for control in controls.values():
        control.observe(update, names='value')
    display(widgets.HBox(list(controls.values())), out)
    update()


@register_cell_magic
def slider(line, cell):
    """%%slider name low high start"""
    parts = line.split()
    name = parts[0]
    low = float(parts[1])
    high = float(parts[2])
    start = float(parts[3]) if len(parts) > 3 else low
    control = widgets.FloatSlider(value=start, min=low, max=high, step=0.1,
                                  description=name, continuous_update=True)
    run_live(cell, {name: control})


@register_cell_magic
def buttons(line, cell):
    """%%buttons name1 name2 ...  (each button is True when pressed)"""
    controls = {}
    for name in line.split():
        controls[name] = widgets.ToggleButton(value=False, description=name,
                                              button_style='info')
    run_live(cell, controls)


print('Lights ready!')


## 1. Recap from Lesson 1: the radon traffic light

Move the slider. Watch which branch of the `if / elif / else` lights up.

**Try it:** what happens at exactly 4.0? At exactly 2.0? Why?

In [ ]:
%%slider reading 0 8 3
if reading > 4:
    light('red', 'POOR: action required')
elif reading >= 2:
    light('gold', 'FAIR')
else:
    light('green', 'GOOD')

## 2. Two buttons, one light: `and` vs `or`

The fan should run only when the room is occupied **and** too hot. Press the buttons.

**Try it:** change `and` to `or`, run the cell again, and press the buttons. What changed?

In [ ]:
%%buttons occupied too_hot
if occupied and too_hot:
    light('limegreen', 'Fan ON')
else:
    light('gray', 'Fan OFF')

## 3. Today: `if` checks once, `while` keeps checking

The ventilation fan keeps running **while** the CO2 level is above 1000 ppm. Each pass through the loop the fan removes 100 ppm. Run the cell and watch.

**Try it:** change the starting `co2` to 900. What happens, and why?

In [ ]:
co2 = 1500   # ppm

while co2 > 1000:
    light('red', 'Fan running... CO2 =', co2, 'ppm')
    co2 -= 100
    wait()

light('limegreen', 'Air OK: CO2 =', co2, 'ppm')